## 4. H&E Patch Feature Extraction

## Overview and Objectives

The objective of this stage is to transform the normalized patches into compact, high-level morphological feature representations. These features will later be aligned with the spatial transcriptomics embeddings to enable joint modeling of image morphology and gene expression.

## Approach

We adopt a multi-scale feature extraction strategy using **EfficientNet-B3** pretrained on ImageNet as the backbone. Features are extracted from three intermediate blocks (blocks 2, 3, and 4) to capture both fine-grained cellular details and coarser tissue architectural patterns. The extracted feature maps are processed through 1×1 convolutions, concatenated, and projected into a 256-dimensional embedding space.



In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## H&E Encoder Function

In [ ]:
# ============================================================
# HEEncoder: Multi-Scale H&E Feature Extractor
# ============================================================

import torch
import torch.nn as nn
import timm

class HEEncoder(nn.Module):
    """
    H&E Encoder using EfficientNet-B3.
    Extracts multi-scale features from stages 2, 3, and 4.
    """
    def __init__(self, embed_dim: int = 256):
        super().__init__()

        # Use features_only for more reliable feature extraction
        self.backbone = timm.create_model(
            'efficientnet_b3',
            pretrained=True,
            features_only=True,
            out_indices=[2, 3, 4]   # Stages 2, 3, 4
        )

        # Get actual output channels from the backbone
        self.feature_channels = self.backbone.feature_info.channels()
        print(f"Feature channels from backbone: {self.feature_channels}")  # For debugging

        # 1x1 convolutions to reduce channels before concatenation
        self.conv1x1_0 = nn.Conv2d(self.feature_channels[0], 64, kernel_size=1)
        self.conv1x1_1 = nn.Conv2d(self.feature_channels[1], 64, kernel_size=1)
        self.conv1x1_2 = nn.Conv2d(self.feature_channels[2], 64, kernel_size=1)

        # Final projection to 256D
        self.projection = nn.Sequential(
            nn.Linear(64 * 3, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(512, embed_dim)
        )

        self.embed_dim = embed_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, 3, 224, 224)
        Returns:
            embedding: (B, embed_dim)
        """
        # Get feature maps from stages 2, 3, and 4
        features = self.backbone(x)   # Returns list of 3 feature maps

        # Apply 1x1 convolutions
        f0 = self.conv1x1_0(features[0])
        f1 = self.conv1x1_1(features[1])
        f2 = self.conv1x1_2(features[2])

        # Global Average Pooling
        f0 = torch.mean(f0, dim=[2, 3])  # (B, 64)
        f1 = torch.mean(f1, dim=[2, 3])  # (B, 64)
        f2 = torch.mean(f2, dim=[2, 3])  # (B, 64)

        # Concatenate
        combined = torch.cat([f0, f1, f2], dim=1)  # (B, 192)

        # Project to 256D
        embedding = self.projection(combined)

        # L2 Normalization
        embedding = nn.functional.normalize(embedding, p=2, dim=1)

        return embedding

### Verification Code (Single Sample Test)

In [ ]:
# ============================================================
# Verification: Test HEEncoder on a single sample
# ============================================================

import torch
import h5py
from pathlib import Path

# ====================== CONFIGURATION ======================
# Pick one sample from your train split
SAMPLE_NAME = "GSE203612_GSM6177599"   # ← Change to any sample you have

PROCESSED_PATCH_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/normalized_patches/train")
sample_h5_path = PROCESSED_PATCH_DIR / f"{SAMPLE_NAME}.h5"
# ============================================================

print(f"Testing on sample: {SAMPLE_NAME}")
print(f"H5 file path: {sample_h5_path}")

# Load patches from .h5 file
with h5py.File(sample_h5_path, 'r') as f:
    patches = f['patches'][:]          # shape: (N, 224, 224, 3)
    coords = f['coords'][:]            # shape: (N, 2)

print(f"Number of patches: {patches.shape[0]}")
print(f"Patch shape: {patches.shape[1:]}")

# Convert to torch tensor and change to (N, 3, 224, 224)
patches_tensor = torch.from_numpy(patches).float().permute(0, 3, 1, 2) / 255.0
print(f"Tensor shape after permutation: {patches_tensor.shape}")

# Initialize the encoder
encoder = HEEncoder(embed_dim=256)
encoder.eval()

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = encoder.to(device)
patches_tensor = patches_tensor.to(device)

print(f"\nRunning feature extraction on device: {device}")

# Extract features (in batches to avoid memory issues)
batch_size = 32
all_features = []

with torch.no_grad():
    for i in range(0, len(patches_tensor), batch_size):
        batch = patches_tensor[i:i+batch_size]
        features = encoder(batch)
        all_features.append(features.cpu())

# Concatenate all features
features_tensor = torch.cat(all_features, dim=0)

print(f"\n✅ Feature extraction successful!")
print(f"Output feature shape: {features_tensor.shape}")          # Should be (N, 256)
print(f"Coordinates shape: {coords.shape}")                      # Should be (N, 2)

# Optional: Check if L2 normalization was applied
print(f"Mean L2 norm of features: {torch.norm(features_tensor, dim=1).mean():.4f}")  # Should be close to 1.0

Testing on sample: GSE203612_GSM6177599
H5 file path: /content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/normalized_patches/train/GSE203612_GSM6177599.h5
Number of patches: 2360
Patch shape: (224, 224, 3)
Tensor shape after permutation: torch.Size([2360, 3, 224, 224])


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Feature channels from backbone: [48, 136, 384]

Running feature extraction on device: cuda

✅ Feature extraction successful!
Output feature shape: torch.Size([2360, 256])
Coordinates shape: (2360, 2)
Mean L2 norm of features: 1.0000


### Full Extraction Script

In [ ]:
# ============================================================
# Full H&E Feature Extraction Pipeline
# ============================================================

import torch
import h5py
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# ====================== CONFIGURATION ======================
# Paths
PROCESSED_PATCHES_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/normalized_patches")
OUTPUT_FEATURES_DIR   = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/HE_Features")

# Splits to process
splits = ["train", "val", "test"]

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize the encoder
encoder = HEEncoder(embed_dim=256)
encoder = encoder.to(device)
encoder.eval()

print("HEEncoder initialized successfully.\n")
# ============================================================

# Create output directories
for split in splits:
    (OUTPUT_FEATURES_DIR / split).mkdir(parents=True, exist_ok=True)

# Process each split
for split in splits:
    print(f"\n{'='*60}")
    print(f"Processing {split.upper()} split")
    print('='*60)

    split_patch_dir = PROCESSED_PATCHES_DIR / split
    split_output_dir = OUTPUT_FEATURES_DIR / split

    h5_files = sorted(list(split_patch_dir.glob("*.h5")))
    print(f"Found {len(h5_files)} samples in {split}")

    for h5_path in tqdm(h5_files, desc=f"Extracting features ({split})"):
        sample_name = h5_path.stem
        output_path = split_output_dir / f"{sample_name}.pt"

        # Skip if already processed
        if output_path.exists():
            continue

        try:
            # Load patches and coordinates
            with h5py.File(h5_path, 'r') as f:
                patches = f['patches'][:]          # (N, 224, 224, 3)
                coords = f['coords'][:]            # (N, 2)

            # Convert to tensor: (N, 3, 224, 224)
            patches_tensor = torch.from_numpy(patches).float().permute(0, 3, 1, 2) / 255.0
            patches_tensor = patches_tensor.to(device)

            # Extract features in batches
            batch_size = 64
            all_features = []

            with torch.no_grad():
                for i in range(0, len(patches_tensor), batch_size):
                    batch = patches_tensor[i:i + batch_size]
                    features = encoder(batch)
                    all_features.append(features.cpu())

            # Concatenate features
            features_tensor = torch.cat(all_features, dim=0)  # (N, 256)
            coords_tensor = torch.from_numpy(coords).float()   # (N, 2)

            # Save as .pt file
            torch.save({
                'features': features_tensor,
                'coords': coords_tensor,
                'sample_name': sample_name,
                'num_patches': features_tensor.shape[0]
            }, output_path)

        except Exception as e:
            print(f"\nError processing {sample_name}: {e}")
            continue

print("\n" + "="*60)
print("FEATURE EXTRACTION COMPLETE")
print("="*60)
print(f"Features saved to: {OUTPUT_FEATURES_DIR}")

Using device: cuda


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Feature channels from backbone: [48, 136, 384]
HEEncoder initialized successfully.


Processing TRAIN split
Found 150 samples in train


Extracting features (train): 100%|██████████| 150/150 [19:13<00:00,  7.69s/it]



Processing VAL split
Found 27 samples in val


Extracting features (val): 100%|██████████| 27/27 [01:10<00:00,  2.63s/it]



Processing TEST split
Found 25 samples in test


Extracting features (test): 100%|██████████| 25/25 [05:18<00:00, 12.74s/it]


FEATURE EXTRACTION COMPLETE
Features saved to: /content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/HE_Features


### Verification of Extracted H&E Feaures

In [ ]:
# ============================================================
# Verification of Extracted H&E Features
# ============================================================

import torch
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ====================== PATHS ======================
FEATURES_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/HE_Features")

splits = ["train", "val", "test"]
# ====================================================

records = []

print("Verifying extracted H&E features...\n")

for split in splits:
    split_dir = FEATURES_DIR / split
    feature_files = sorted(list(split_dir.glob("*.pt")))

    print(f"\n{'='*60}")
    print(f"Split: {split.upper()} | Samples: {len(feature_files)}")
    print('='*60)

    for file_path in tqdm(feature_files, desc=f"Checking {split}"):
        try:
            data = torch.load(file_path, map_location="cpu")

            features = data['features']
            coords = data['coords']
            sample_name = data.get('sample_name', file_path.stem)
            num_patches = data.get('num_patches', features.shape[0])

            record = {
                'split': split,
                'sample_name': sample_name,
                'num_patches': num_patches,
                'feature_dim': features.shape[1],
                'feature_mean': features.mean().item(),
                'feature_std': features.std().item(),
                'feature_min': features.min().item(),
                'feature_max': features.max().item(),
                'l2_norm_mean': torch.norm(features, dim=1).mean().item()
            }
            records.append(record)

        except Exception as e:
            print(f"\nError loading {file_path.name}: {e}")

# Create summary DataFrame
df = pd.DataFrame(records)

print("\n" + "="*70)
print("H&E FEATURE EXTRACTION VERIFICATION SUMMARY")
print("="*70)

print("\n--- Per Split Statistics ---")
summary = df.groupby('split').agg({
    'sample_name': 'count',
    'num_patches': ['mean', 'min', 'max'],
    'feature_dim': 'first',
    'l2_norm_mean': 'mean'
}).round(2)

summary.columns = ['Num_Samples', 'Avg_Patches', 'Min_Patches', 'Max_Patches', 'Feature_Dim', 'Avg_L2_Norm']
print(summary)

print("\n--- Overall Feature Statistics ---")
print(f"Total samples processed : {len(df)}")
print(f"Overall mean feature value : {df['feature_mean'].mean():.4f}")
print(f"Overall std of features    : {df['feature_std'].mean():.4f}")
print(f"Average L2 norm            : {df['l2_norm_mean'].mean():.4f}")

# Check for any anomalies
print("\n--- Quality Checks ---")
print(f"Samples with feature_dim != 256 : {(df['feature_dim'] != 256).sum()}")
print(f"Samples with L2 norm far from 1 : {(df['l2_norm_mean'] < 0.95).sum()}")

Verifying extracted H&E features...


Split: TRAIN | Samples: 150


Checking train: 100%|██████████| 150/150 [00:01<00:00, 120.46it/s]



Split: VAL | Samples: 27


Checking val: 100%|██████████| 27/27 [00:00<00:00, 283.90it/s]



Split: TEST | Samples: 25


Checking test: 100%|██████████| 25/25 [00:00<00:00, 157.00it/s]



H&E FEATURE EXTRACTION VERIFICATION SUMMARY

--- Per Split Statistics ---
       Num_Samples  Avg_Patches  Min_Patches  Max_Patches  Feature_Dim  \
split                                                                    
test            25      2132.52          188         3187          256   
train          150       919.94          131         4865          256   
val             27       301.67          114          703          256   

       Avg_L2_Norm  
split               
test           1.0  
train          1.0  
val            1.0  

--- Overall Feature Statistics ---
Total samples processed : 202
Overall mean feature value : -0.0003
Overall std of features    : 0.0625
Average L2 norm            : 1.0000

--- Quality Checks ---
Samples with feature_dim != 256 : 0
Samples with L2 norm far from 1 : 0


### UMAP Projection (One Sample Per Split)

In [ ]:
# ============================================================
# UMAP VISUALIZATION FOR H&E FEATURES
# ============================================================

import umap
import matplotlib.pyplot as plt
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# ─── Configuration ─────────────────────────────────────────────
PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")

HE_FEATURES_DIR = PROJECT_ROOT / "data" / "processed" / "HE_Features"
VISUALIZATION_DIR = PROJECT_ROOT / "visualizations" / "umap"

# Recommended improved UMAP parameters
UMAP_N_NEIGHBORS = 30
UMAP_METRIC = "correlation"   # Options: 'cosine', 'correlation', 'euclidean'

print(f"UMAP Settings: n_neighbors={UMAP_N_NEIGHBORS}, metric='{UMAP_METRIC}'")


# ─── Improved UMAP Function ────────────────────────────────────
def generate_umap_for_sample(sample_name: str, split: str,
                             n_neighbors: int = UMAP_N_NEIGHBORS,
                             metric: str = UMAP_METRIC):
    """
    Generate a high-quality UMAP plot for one sample's H&E features.
    """
    feature_path = HE_FEATURES_DIR / split / f"{sample_name}.pt"

    if not feature_path.exists():
        print(f"❌ Feature file not found: {feature_path}")
        return None

    # Load features
    data = torch.load(feature_path, map_location="cpu")
    features = data["features"].numpy()          # shape: (N_patches, 256)
    coords = data.get("coords", None)

    print(f"Processing: {sample_name} | Split: {split.upper()} | Patches: {len(features)}")

    # Run UMAP with improved parameters
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=0.1,
        metric=metric,
        random_state=42,
        n_components=2,
        verbose=False
    )

    embedding = reducer.fit_transform(features)

    # Create plot
    plt.figure(figsize=(9, 7))
    scatter = plt.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=np.arange(len(embedding)),   # Color by patch order (can change to pseudo-label later)
        cmap="viridis",
        s=12,
        alpha=0.75,
        edgecolors="none"
    )

    plt.title(
        f"UMAP of H&E Features\n{sample_name} ({split.upper()})",
        fontsize=13, pad=15
    )
    plt.xlabel("UMAP 1", fontsize=11)
    plt.ylabel("UMAP 2", fontsize=11)

    cbar = plt.colorbar(scatter, shrink=0.8)
    cbar.set_label("Patch Index", fontsize=10)

    plt.grid(True, alpha=0.25, linestyle="--")
    plt.tight_layout()

    # Save figure
    save_dir = VISUALIZATION_DIR / split
    save_dir.mkdir(parents=True, exist_ok=True)

    save_path = save_dir / f"umap_{sample_name}_n{n_neighbors}_{metric}.png"
    plt.savefig(save_path, dpi=1200, bbox_inches="tight", facecolor="white")
    plt.close()

    print(f"✅ Saved improved UMAP → {save_path}\n")
    return embedding


# ============================================================
# EXAMPLE: Run on one sample from each split
# ============================================================
# You can change these sample names to any sample you want to visualize

samples_to_visualize = {
    "train": "GSE213688_GSM6592059",           # Example Train sample
    "val":   "Human_Breast_Andersson_10142021_ST_B3",  # Example Val sample
    "test":  "GSE212482_GSM6543813",         # Example Test sample
}

print("=" * 70)
print("GENERATING IMPROVED UMAPs FOR ALL SPLITS")
print("=" * 70)

for split_name, sample_name in samples_to_visualize.items():
    generate_umap_for_sample(
        sample_name=sample_name,
        split=split_name,
        n_neighbors=UMAP_N_NEIGHBORS,
        metric=UMAP_METRIC
    )

print("\n🎉 All UMAP visualizations completed!")

UMAP Settings: n_neighbors=30, metric='correlation'
GENERATING IMPROVED UMAPs FOR ALL SPLITS
Processing: GSE213688_GSM6592059 | Split: TRAIN | Patches: 2498
✅ Saved improved UMAP → /content/drive/My Drive/MSC Project/SpaHisto-Net/visualizations/umap/train/umap_GSE213688_GSM6592059_n30_correlation.png

Processing: Human_Breast_Andersson_10142021_ST_B3 | Split: VAL | Patches: 153
✅ Saved improved UMAP → /content/drive/My Drive/MSC Project/SpaHisto-Net/visualizations/umap/val/umap_Human_Breast_Andersson_10142021_ST_B3_n30_correlation.png

Processing: GSE212482_GSM6543813 | Split: TEST | Patches: 1659
✅ Saved improved UMAP → /content/drive/My Drive/MSC Project/SpaHisto-Net/visualizations/umap/test/umap_GSE212482_GSM6543813_n30_correlation.png


🎉 All UMAP visualizations completed!


In [ ]:
pip install grad-cam

In [ ]:
# ============================================================
# Grad-CAM Visualization for H&E Features
# ============================================================

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import h5py
import warnings
warnings.filterwarnings("ignore")


# ====================== CONFIGURATION ======================
FEATURES_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/HE_Features")
PATCHES_DIR  = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/normalized_patches")

# Number of patches to visualize per sample
NUM_PATCHES_PER_SAMPLE = 5

# Output directory
SAVE_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/visualizations/gradcam")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the encoder
encoder = HEEncoder(embed_dim=256)
encoder = encoder.to(device)
encoder.eval()

# We will use the last convolutional layer for Grad-CAM
# In our HEEncoder, this corresponds to conv1x1_block4
target_layers = [encoder.conv1x1_2]

# Initialize GradCAM
cam = GradCAM(model=encoder, target_layers=target_layers)

# ====================== SELECT SAMPLES ======================
# Pick samples with relatively high Tumor density (you can adjust these)
samples_for_gradcam = {
    "train": "GSE213688_GSM6592059",
    "val":   "Human_Breast_Andersson_10142021_ST_B3",
    "test":  "GSE212482_GSM6543813"
}
# ============================================================

for split, sample_name in samples_for_gradcam.items():
    print(f"\nGenerating Grad-CAM for {split.upper()}: {sample_name}")

    h5_path = PATCHES_DIR / split / f"{sample_name}.h5"

    if not h5_path.exists():
        print(f"  H5 file not found. Skipping.")
        continue

    # Load patches
    with h5py.File(h5_path, 'r') as f:
        patches = f['patches'][:]          # (N, 224, 224, 3)
        coords = f['coords'][:]

    # Randomly select patches for visualization
    indices = np.random.choice(len(patches), NUM_PATCHES_PER_SAMPLE, replace=False)

    for idx in indices:
        patch = patches[idx]                    # (224, 224, 3)
        patch_tensor = torch.from_numpy(patch).float().permute(2, 0, 1).unsqueeze(0) / 255.0
        patch_tensor = patch_tensor.to(device)

        # Generate Grad-CAM
        grayscale_cam = cam(input_tensor=patch_tensor, targets=None)
        grayscale_cam = grayscale_cam[0, :]

        # Overlay CAM on original image
        visualization = show_cam_on_image(patch / 255.0, grayscale_cam, use_rgb=True)

        # Plot
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))

        axes[0].imshow(patch)
        axes[0].set_title("Original Patch")
        axes[0].axis('off')

        axes[1].imshow(visualization)
        axes[1].set_title("Grad-CAM")
        axes[1].axis('off')

        plt.suptitle(f"{sample_name} | Patch {idx}", fontsize=12, fontweight='bold')
        plt.tight_layout()

        # Save
        save_path = SAVE_DIR / split / sample_name
        save_path.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path / f"gradcam_patch_{idx}.png", dpi=1200, bbox_inches='tight')
        plt.close()

print("\n✅ Grad-CAM visualizations completed.")

Feature channels from backbone: [48, 136, 384]

Generating Grad-CAM for TRAIN: GSE213688_GSM6592059

Generating Grad-CAM for VAL: Human_Breast_Andersson_10142021_ST_B3

Generating Grad-CAM for TEST: GSE212482_GSM6543813

✅ Grad-CAM visualizations completed.


# 5. Spatial Transcriptomics Feature Extraction

## 5.1 Overview and Objectives

To extract rich, low-dimensional embeddings from the spatial transcriptomics (ST) data. While the H&E branch captures morphological patterns from image patches, the ST branch must capture both **molecular information** (gene expression) and **spatial context** (relationships between neighboring spots).

The goal of this stage is to produce **256-dimensional embeddings** per spot that are:
- Biologically meaningful
- Spatially aware
- Compatible in dimension with the H&E features for effective multimodal fusion

## 5.2 Approach

We implement a **Graph Attention Network (GAT)** encoder that operates directly on the spatial KNN graph (k=6) already constructed during preprocessing. GAT is chosen over GraphSAGE because it can learn **attention weights** — allowing the model to assign higher importance to more relevant neighboring spots. This is particularly valuable in heterogeneous tumor microenvironments where not all neighbors contribute equally to a spot’s molecular profile.

Training is performed in a **semi-supervised** manner using the high-quality pseudo-labels (Tumor, Tumor Stroma, Non-Tumor) generated in the previous stage. Spots labeled as “Uncertain” are excluded from the classification loss to reduce noise propagation.


In [2]:
from pathlib import Path

# Define paths
PROJECT_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net")
models_dir = PROJECT_ROOT / "src" / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Create __init__.py if it doesn't exist
init_file = models_dir / "__init__.py"
if not init_file.exists():
    init_file.touch()

print(f"✅ Created folder: {models_dir}")

✅ Created folder: /content/drive/My Drive/MSC Project/SpaHisto-Net/src/models


In [3]:
module_content = '''"""
STEncoder: Graph Attention Network for Spatial Transcriptomics Feature Extraction
SpaHisto-Net Project
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.data import Data, Batch
from typing import Optional, Tuple


class STEncoder(nn.Module):
    """
    Graph Attention Network (GAT) encoder for spatial transcriptomics.

    Takes gene expression + spatial graph → 256D embedding per spot.
    Designed to match the 256D output dimension of the H&E encoder.
    """

    def __init__(self,
                 in_channels: int = 2000,
                 hidden_channels: int = 256,
                 out_channels: int = 256,
                 num_heads: int = 4,
                 dropout: float = 0.2):
        super().__init__()

        # GAT layers with multi-head attention
        self.gat1 = GATConv(in_channels, hidden_channels, heads=num_heads, dropout=dropout)
        self.gat2 = GATConv(hidden_channels * num_heads, hidden_channels, heads=num_heads, dropout=dropout)
        self.gat3 = GATConv(hidden_channels * num_heads, hidden_channels, heads=1, dropout=dropout)

        # Projection head to final 256D embedding
        self.projection = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, out_channels)
        )

        self.norm = nn.LayerNorm(out_channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
                batch: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Forward pass.

        Args:
            x: Node features [num_spots, in_channels]
            edge_index: Graph connectivity [2, num_edges]
            batch: Batch vector for multiple graphs (optional)

        Returns:
            256-dimensional embeddings (L2 normalized)
        """
        # GAT layers with ELU activation
        x = F.elu(self.gat1(x, edge_index))
        x = self.dropout(x)

        x = F.elu(self.gat2(x, edge_index))
        x = self.dropout(x)

        x = self.gat3(x, edge_index)

        # Global pooling if processing multiple samples at once
        if batch is not None:
            x = global_mean_pool(x, batch)

        # Final projection + normalization
        x = self.projection(x)
        x = self.norm(x)
        x = F.normalize(x, p=2, dim=-1)   # L2 normalize (consistent with H&E)

        return x


def create_st_data_from_anndata(adata) -> Data:
    """
    Helper function to convert AnnData to PyTorch Geometric Data object.
    """
    from torch_geometric.utils import from_scipy_sparse_matrix

    X = torch.tensor(adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X,
                     dtype=torch.float32)

    edge_index = from_scipy_sparse_matrix(adata.obsp["spatial_connectivities"])[0]

    return Data(x=X, edge_index=edge_index)
'''

# Write the module
module_path = models_dir / "st_encoder.py"
with open(module_path, "w") as f:
    f.write(module_content)

print(f"✅ STEncoder module saved to: {module_path}")

✅ STEncoder module saved to: /content/drive/My Drive/MSC Project/SpaHisto-Net/src/models/st_encoder.py
